# S14 · Pull real market data (with an offline fallback)

We start the finance block from zero. First a one-line explanation of every market
word you'll hear today. Then we pull a real Indian market index into Python, and if
there's no internet we build a realistic price series ourselves so the notebook
always runs. Finally we look at what market data is and plot the price.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to finance? Perfect — that's who this is written for. Every market word is
  explained in one line the first time it appears.
- Want the key idea in advance? Open `primers/returns_and_log_returns.md`.
- Already trade or already know markets? Skip to the cell marked **Stretch
  (optional)** near the end.
- Stuck on a word? It's in `primers/glossary.md`.

## First, the words (no maths, just plain English)

Finance has a lot of jargon that hides simple ideas. Here is the whole vocabulary
you need for today, one plain line each:

- **Market** — a place where buyers and sellers agree on a price. In India the big
  stock exchanges are the NSE and BSE, both in Mumbai.
- **Share (equity)** — owning a share means you own a tiny slice of a company. If
  you hold one Reliance share, you own a (very small) piece of Reliance.
- **Bond** — a loan you give to a company or a government; they pay you interest and
  return the amount at the end. Safer than shares, usually smaller returns.
- **Derivative** — a contract whose value comes *from* something else, like a bet on
  where a stock will be next month. Options and futures are derivatives.
- **Index** — one number that tracks a whole basket of stocks. The **Nifty 50** is a
  basket that tracks India's 50 biggest listed companies; the **Sensex** tracks 30.
  An index is the market's thermometer.
- **Going long** — buying now, hoping the price rises. The ordinary way to invest.
- **Going short** — selling first (with a borrowed asset), hoping to buy it back
  cheaper later. You profit when the price *falls*.
- **Risk vs return** — the one rule of the game: there is no return without risk.
  Anything that might pay more can also lose more. A savings account is safe and
  dull; a small stock is risky and might soar or sink.

The single most important habit to build: prices are **uncertain**. Nobody sets
tomorrow's price. It emerges from a crowd of buy and sell decisions, so it is
genuinely random. That is why this whole module reaches for probability.

## Setup

If you are on **Google Colab**, run the next cell once. On your **own machine** you
already installed everything with `uv`, so the next cell does nothing there.

In [ ]:
# This notebook uses numpy, pandas and matplotlib (all of which Colab already
# has) plus yfinance, which Colab does not ship. So we install only yfinance,
# and only when we are actually on Colab.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import numpy as np                  # fast maths on lists of numbers
import pandas as pd                   # tables of data, indexed by date
import matplotlib.pyplot as plt       # drawing charts

# A seed makes any random numbers (our offline fallback) identical every run.
np.random.seed(0)

## Step 1 — get a price series

We try to **download** real data with `yfinance`, a free library that pulls prices
from Yahoo Finance. We ask for the **Nifty 50** (symbol `^NSEI`), the index that
tracks India's 50 largest companies.

If the download fails, say because you're offline, we quietly build a **synthetic**
(made-up but realistic) price series with NumPy instead, so this notebook always
works. Either way we end up with `close_price`: the daily **closing price**, meaning
the last traded price of each day, as a pandas Series indexed by date.

Don't worry about the details of the fallback yet; the important thing is that the
`try` block downloads real data and the `except` block invents data only if the
download fails.

In [ ]:
# The ticker (symbol) we try to download. "^NSEI" is the Nifty 50. You could
# also try "RELIANCE.NS" for the Reliance share on the NSE, or "TCS.NS" for TCS.
ticker_symbol = "^NSEI"

# We will end up with `close_price`, and note where the data came from.
close_price = None
data_source = ""

try:
    # yfinance downloads market data from Yahoo Finance.
    import yfinance as yf

    # Download about four years of daily data. auto_adjust=True gives the
    # "adjusted" close, which corrects for stock splits and dividends so the
    # numbers reflect what an investor really experienced.
    downloaded = yf.download(
        ticker_symbol,
        start="2021-01-01",
        end="2025-01-01",
        auto_adjust=True,
        progress=False,
    )

    # If the download came back empty (no internet, or the symbol failed), raise
    # an error so we drop into the offline fallback below.
    if downloaded is None or len(downloaded) == 0:
        raise ValueError("no data returned")

    # Keep just the closing price and drop any missing days.
    close_price = downloaded["Close"].dropna()

    # Some yfinance versions return a one-column table; squeeze it to a plain
    # Series so the rest of the notebook stays simple.
    close_price = close_price.squeeze()

    data_source = "LIVE download from yfinance (" + ticker_symbol + ")"

except Exception:
    # OFFLINE FALLBACK ----------------------------------------------------
    # No internet (or the download failed), so we BUILD a realistic price series
    # ourselves. This is a "geometric random walk": each day the price is
    # multiplied by exp(a small random number). That keeps the price positive
    # and makes it wander and trend like a real stock.
    print("Could not download live data; using a synthetic price series instead.")

    number_of_days = 1000        # about four trading years
    daily_drift = 0.0003         # tiny average upward push per day
    daily_volatility = 0.012     # how much the price typically swings per day

    # One random "shock" per day, from a normal (bell-curve) distribution.
    daily_shocks = np.random.normal(daily_drift, daily_volatility, size=number_of_days)

    # Price = starting level times exp of the running sum of the shocks.
    starting_price = 15000.0
    price_levels = starting_price * np.exp(np.cumsum(daily_shocks))

    # Give it real-looking business-day dates so it behaves like live data.
    dates = pd.bdate_range(start="2021-01-01", periods=number_of_days)
    close_price = pd.Series(price_levels, index=dates, name="Close")

    data_source = "SYNTHETIC fallback (seeded geometric random walk)"

print("Data source:", data_source)
print("Number of days:", len(close_price))

## Step 2 — look at the first few prices

Always look at your data before doing anything with it. Each row is one trading day;
the value is that day's closing price. `.head()` shows the first five rows.

In [ ]:
print(close_price.head())

## Step 3 — what is OHLCV?

Real market data usually comes with **five numbers for each day**, known together as
OHLCV:

- **Open** — the first traded price of the day.
- **High** — the highest price during the day.
- **Low** — the lowest price during the day.
- **Close** — the last traded price of the day.
- **Volume** — how many shares changed hands that day.

We model the **Close** (specifically the *adjusted* close). Below we build a tiny
hand-made OHLCV table just so you can see the shape of it.

In [ ]:
# A small hand-made example so you can see what OHLCV looks like.
# (This is only for illustration; we do not use it later.)
example_ohlcv = pd.DataFrame({
    "Open":   [100.0, 106.0, 104.0],
    "High":   [108.0, 110.0, 107.0],
    "Low":    [ 98.0, 103.0, 100.0],
    "Close":  [106.0, 104.0, 101.0],
    "Volume": [12000,  9500, 11000],
})

print(example_ohlcv)

## Step 4 — plot the close price

Now we plot our real (or synthetic) close price over time. Notice the shape: a price
series tends to **wander and trend**. It drifts up and down with no fixed level it
returns to. Keep this wandering picture in mind — the whole next notebook is about
turning this restless line into something steady.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(close_price.index, close_price.values, color="#2E75B6")
plt.xlabel("date")
plt.ylabel("closing price")
plt.title("Daily closing price: it wanders and trends")
plt.show()

## Step 5 — a quick numeric summary

Three numbers give a feel for the series: its lowest close, its highest close, and
its average. pandas has these built in.

In [ ]:
print("Lowest close :", round(float(close_price.min()), 2))
print("Highest close:", round(float(close_price.max()), 2))
print("Average close:", round(float(close_price.mean()), 2))

### Stretch (optional) — pull a single stock instead of the index

Skip this if you're new to code. If you're comfortable, change the target from the
Nifty index to one company's share and see the same pipeline work. This cell tries a
live download of Reliance; if you're offline it just reports that and moves on,
leaving the main `close_price` above untouched.

In [ ]:
# Try a single NSE stock instead of the whole index. "RELIANCE.NS" is the
# Reliance Industries share; "TCS.NS", "INFY.NS", "HDFCBANK.NS" also work.
stock_symbol = "RELIANCE.NS"

try:
    import yfinance as yf
    stock_data = yf.download(stock_symbol, start="2023-01-01", end="2025-01-01",
                             auto_adjust=True, progress=False)
    if stock_data is None or len(stock_data) == 0:
        raise ValueError("no data returned")
    stock_close = stock_data["Close"].dropna().squeeze()
    print("Downloaded", len(stock_close), "days for", stock_symbol)
    print("Latest close:", round(float(stock_close.iloc[-1]), 2))
except Exception:
    print("Offline (or the symbol failed) - skipping the single-stock download.")
    print("The main Nifty series above still works, so the rest runs fine.")

## What you just did

You learned the finance vocabulary for the whole session, pulled a real Indian
market price series into Python (or built a realistic one offline), met the OHLCV
shape of market data, and plotted the close price. You saw that a price *wanders*.

Next notebook: `02_returns_and_stylized_facts.ipynb`, where we turn this wandering
price into **returns** — the steady, comparable quantity we actually model — and see
what real markets look like up close.